# Teams history generator

Load every match JSON file, validate its structure and columns, merge all seasons, and store the result in `data/teams_history.csv`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

JSON_DIR = Path("../json")
OUTPUT_PATH = Path("../data/teams_history.csv")

## Discover and validate JSON files

In [ ]:
json_files = sorted(JSON_DIR.glob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {JSON_DIR.resolve()}")

pd.DataFrame({"json_file": [path.name for path in json_files]})

In [ ]:
payloads = {}
validation_errors = []

for json_path in json_files:
    try:
        with json_path.open(encoding="utf-8") as json_file:
            payload = json.load(json_file)

        if not isinstance(payload, dict):
            raise TypeError("The root element must be an object")
        if not isinstance(payload.get("matches"), list):
            raise TypeError("The 'matches' field must be a list")

        payloads[json_path.name] = payload
    except (json.JSONDecodeError, OSError, TypeError) as error:
        validation_errors.append({"json_file": json_path.name, "error": str(error)})

if validation_errors:
    display(pd.DataFrame(validation_errors))
    raise ValueError("Invalid JSON files found. Fix them before generating teams_history.csv.")

print(f"Validated {len(payloads)} JSON files.")

## Convert every JSON file to a DataFrame

In [ ]:
dataframes = {}

for json_file_name, payload in payloads.items():
    matches_df = pd.json_normalize(payload["matches"], sep="_")
    matches_df.insert(0, "competition", payload.get("name", ""))
    matches_df.insert(1, "source_file", json_file_name)
    dataframes[json_file_name] = matches_df

pd.DataFrame(
    {
        "json_file": file_name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
    }
    for file_name, dataframe in dataframes.items()
)

## Check column correspondence

In [ ]:
reference_file = next(iter(dataframes))
reference_columns = dataframes[reference_file].columns.tolist()
reference_column_set = set(reference_columns)

column_checks = []
for file_name, dataframe in dataframes.items():
    current_columns = dataframe.columns.tolist()
    current_column_set = set(current_columns)
    column_checks.append(
        {
            "json_file": file_name,
            "same_columns": current_column_set == reference_column_set,
            "same_order": current_columns == reference_columns,
            "missing_columns": sorted(reference_column_set - current_column_set),
            "extra_columns": sorted(current_column_set - reference_column_set),
        }
    )

column_check_df = pd.DataFrame(column_checks)
display(column_check_df)

if not column_check_df[["same_columns", "same_order"]].all(axis=None):
    raise ValueError("The JSON DataFrames do not have matching ordered columns.")

## Merge and inspect the complete history

In [ ]:
teams_history = pd.concat(dataframes.values(), ignore_index=True)
teams_history = teams_history.sort_values(["date", "time", "team1", "team2"]).reset_index(drop=True)

print(f"Rows: {len(teams_history):,}")
print(f"Columns: {len(teams_history.columns)}")
display(teams_history.head())
display(teams_history.tail())

## Store `teams_history.csv`

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
teams_history.to_csv(OUTPUT_PATH, index=False)

print(f"Stored {len(teams_history):,} rows in {OUTPUT_PATH.resolve()}")